<h3> Tabela de churn por segmento

In [0]:
%sql
CREATE OR REPLACE TABLE projeto_churn.gold.churn_por_segmento AS
SELECT 
    Contract AS tipo_contrato,
    InternetService AS servico_internet,
    faixa_tenure,
    COUNT(*) AS qtd_clientes,
    SUM(churn_flag) AS qtd_churn,
    ROUND(AVG(churn_flag) * 100, 2) AS taxa_churn_pct,
    ROUND(SUM(MonthlyCharges), 2) AS receita_mensal_total,
    ROUND(SUM(CASE WHEN churn_flag = 1 THEN MonthlyCharges ELSE 0 END), 2) AS receita_em_risco
FROM projeto_churn.silver.clientes
GROUP BY 1, 2, 3;


num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT * 
FROM projeto_churn.gold.churn_por_segmento 
ORDER BY receita_em_risco DESC;

tipo_contrato,servico_internet,faixa_tenure,qtd_clientes,qtd_churn,taxa_churn_pct,receita_mensal_total,receita_em_risco
Month-to-month,Fiber optic,0-6 meses,619,459,74.15,49908.8,37150.15
Month-to-month,Fiber optic,25-48 meses,521,226,43.38,47535.45,20762.05
Month-to-month,Fiber optic,13-24 meses,425,215,50.59,37205.1,19046.9
Month-to-month,Fiber optic,7-12 meses,297,184,61.95,25274.35,16028.15
Month-to-month,DSL,0-6 meses,495,245,49.49,22999.2,10999.65
Month-to-month,Fiber optic,49+ meses,266,78,29.32,25257.4,7494.75
One year,Fiber optic,49+ meses,355,67,18.87,35673.5,6926.2
One year,Fiber optic,25-48 meses,154,31,20.13,14790.75,3039.1
Two year,Fiber optic,49+ meses,393,28,7.12,41197.15,2965.55
Month-to-month,DSL,13-24 meses,232,55,23.71,12221.9,2765.4


<h3> Tabela de valor do cliente

In [0]:
%sql
CREATE OR REPLACE TABLE projeto_churn.gold.valor_cliente AS
SELECT
    customerID,
    Contract AS tipo_contrato,
    tenure AS meses_de_casa,
    MonthlyCharges AS mensalidade,
    TotalCharges AS receita_acumulada,
    churn_flag,
    ROUND(MonthlyCharges * CASE
        WHEN Contract = 'Two year' THEN 36
        WHEN Contract = 'One year' THEN 24
        ELSE 12 END, 2) AS ltv_projetado
FROM projeto_churn.silver.clientes;

num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT *
FROM projeto_churn.gold.valor_cliente
ORDER BY ltv_projetado DESC 
LIMIT 10;

customerID,tipo_contrato,meses_de_casa,mensalidade,receita_acumulada,churn_flag,ltv_projetado
7569-NMZYQ,Two year,72,118.75,8672.45,0,4275.0
8984-HPEMB,Two year,71,118.65,8477.6,0,4271.4
5989-AXPUC,Two year,68,118.6,7990.05,0,4269.6
9924-JPRMC,Two year,72,118.2,8547.15,0,4255.2
3810-DVDQQ,Two year,72,117.6,8308.9,0,4233.6
9739-JLPQJ,Two year,72,117.5,8670.1,0,4230.0
6904-JLBGY,Two year,72,117.35,8436.25,0,4224.6
6650-BWFRT,Two year,72,117.15,8529.5,0,4217.4
9788-HNGUT,Two year,72,116.95,8594.4,0,4210.2
1488-PBLJN,Two year,72,116.85,8477.7,0,4206.6


<h3>Explorar os resultados

In [0]:
%sql
SELECT tipo_contrato,
       SUM(qtd_clientes) AS clientes,
       ROUND(SUM(qtd_churn) / SUM(qtd_clientes) * 100, 2) AS taxa_churn,
       ROUND(SUM(receita_em_risco), 2) AS receita_em_risco
FROM projeto_churn.gold.churn_por_segmento
GROUP BY 1
ORDER BY taxa_churn DESC; 

tipo_contrato,clientes,taxa_churn,receita_em_risco
Month-to-month,3875,42.71,120847.1
One year,1473,11.27,14118.45
Two year,1695,2.83,4165.3
